# Phase 2 | 04. Evaluation Metrics

RMSE measures prediction accuracy — how close our estimated ratings are to actual ratings.
But it says nothing about **recommendation quality**: are we surfacing items users actually care about?
Are we diversifying across the catalog, or just recommending the same 100 blockbusters to everyone?

This notebook implements four industry-standard metrics from scratch:

| Metric | What it measures |
|--------|------------------|
| RMSE | Prediction accuracy (baseline) |
| NDCG@10 | Ranking quality — are relevant items ranked higher? |
| Catalog Coverage | What % of items does the model ever recommend? |
| Long-tail Exposure Rate | How often does the model surface niche (non-popular) items? |

The last two connect directly back to the **popularity bias** finding from Phase 1.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.append('../')
from sklearn.model_selection import train_test_split
from src.mf_model import MatrixFactorization

In [ ]:
# --- data (identical setup to notebook 03) ---
ratings = pd.read_parquet('../data/processed/ratings.parquet')

ratings_small = ratings.sample(frac=0.02, random_state=42)
train, test = train_test_split(ratings_small, test_size=0.2, random_state=42)

train_array = train[['userId', 'movieId', 'rating']].values
test_array  = test[['userId', 'movieId', 'rating']].values

print(f"train: {train_array.shape}, test: {test_array.shape}")

In [ ]:
# --- train model with best params from notebook 03 ---
model = MatrixFactorization(
    n_factors=20,
    lr=0.005,
    lambda_=0.05,
    n_epochs=100,
    patience=5,
    random_state=42,
)
model.fit(train_array, val_ratings=test_array, verbose=True)

model._seen = (
    train.groupby('userId')['movieId']
    .apply(set)
    .to_dict()
)

print(f"\nBest epoch: {model.best_epoch} | Best val RMSE: {min(model.val_losses):.4f}")

## 2. RMSE (baseline)

RMSE is our baseline accuracy metric. It penalises large errors more than small ones (squared term),
and lives on the same scale as ratings (0.5–5.0), making it interpretable: an RMSE of 0.90 means
predictions are off by roughly one star on average.

**Limitation**: RMSE doesn't care about the *order* of predictions. A model that perfectly predicts
every rating but ranks the worst item first would still score well on RMSE.

In [ ]:
def rmse(actual: np.ndarray, predicted: np.ndarray) -> float:
    return np.sqrt(np.mean((actual - predicted) ** 2))


test_actual    = test_array[:, 2]
test_predicted = np.array([
    model.predict(int(row[0]), int(row[1]))
    for row in test_array
])

test_rmse = rmse(test_actual, test_predicted)
print(f"Test RMSE: {test_rmse:.4f}")

## 3. NDCG@K

**Normalised Discounted Cumulative Gain** measures ranking quality.
The core idea: a relevant item appearing at rank 1 is worth more than the same item appearing at rank 10.

**DCG** discounts each item's relevance by the log of its rank:
$$\text{DCG@K} = \sum_{i=1}^{K} \frac{\text{rel}_i}{\log_2(i+1)}$$

**NDCG** normalises by the *ideal* DCG (items sorted by relevance descending), giving a score in [0, 1]:
$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

### Why open-world NDCG fails here

Ranking all ~16k items and checking if the top-10 overlap with a user's handful of test ratings
gives near-zero NDCG by construction — the overlap probability is under 0.1% by chance.
The debug cells below confirm this before switching to the correct approach.

### Correct approach: closed-world evaluation

Score only the items in the user's test set, rank them by model score, and compute NDCG.
The question becomes: *among items this user actually rated, does the model rank the ones
they liked most at the top?* This is a meaningful and answerable question.

In [ ]:
def dcg_at_k(r: list, k: int) -> float:
    r = np.array(r[:k], dtype=float)
    if r.size == 0:
        return 0.0
    return float(np.sum(r / np.log2(np.arange(2, r.size + 2))))


def ndcg_at_k(actual: dict, predicted: list, k: int = 10) -> float:
    """
    actual    : {item_id: rating}
    predicted : list of item_ids ranked by model score (descending)
    """
    r = [actual.get(item, 0.0) for item in predicted[:k]]
    ideal = sorted(actual.values(), reverse=True)
    idcg = dcg_at_k(ideal, k)
    if idcg == 0.0:
        return 0.0
    return dcg_at_k(r, k) / idcg

### Step 1 — Debug: open-world failure

In [ ]:
# build test lookup: user_id -> {item_id: rating}
test_lookup = (
    test.groupby('userId')
    .apply(lambda df: dict(zip(df['movieId'], df['rating'])))
    .to_dict()
)

eligible_users = [
    uid for uid, items in test_lookup.items()
    if uid in model.user2idx and len(items) >= 5
]

np.random.seed(42)
sampled_users = np.random.choice(
    eligible_users,
    size=min(200, len(eligible_users)),
    replace=False,
)

# pick one user and inspect
debug_uid = int(sampled_users[0])
debug_test_items = test_lookup[debug_uid]

print(f"User {debug_uid} — test-rated items: {len(debug_test_items)}")
print(f"  item IDs: {list(debug_test_items.keys())}")

open_recs = model.recommend(debug_uid, top_k=10, exclude_seen=True)
print(f"\nOpen-world top-10 recommendations:")
overlap = 0
for item, score in open_recs:
    tag = " <-- in test set" if item in debug_test_items else ""
    print(f"  item {item:6d}  score={score:.3f}{tag}")
    if item in debug_test_items:
        overlap += 1
print(f"\nOverlap: {overlap}/10")
print(f"Catalog size: {len(model.item2idx):,} items  |  chance overlap ≈ {10/len(model.item2idx)*100:.3f}%")

### Step 2 — Check: how many test items are known to the model?

In [ ]:
# With a random 2% sample + 80/20 split, some items land only in test
# and were never seen during training — model.item2idx won't contain them.
# model.predict() returns the global mean (mu) for unknown items,
# so closed-world evaluation still works — it just has less information for those items.

for uid in sampled_users[:5]:
    test_items = set(test_lookup[uid].keys())
    known = test_items & set(model.item2idx.keys())
    print(f"user {uid}: {len(test_items)} test items, {len(known)} known to model")

### Step 3 — Fix: closed-world NDCG

Score only the items in each user's test set. Rank by model score.
For items unknown to the model, `model.predict()` returns the global mean — a reasonable fallback
that keeps all test items in the candidate set.

This answers: *can the model distinguish which of a user's rated items they actually liked most?*

In [ ]:
K = 10
ndcg_scores = []
recommendations = {}

for uid in sampled_users:
    # Closed-world evaluation: score only items the user rated in test set
    # This is the correct offline NDCG — we're asking "did the model rank
    # the user's good items above their bad items?"
    candidate_items = list(test_lookup[uid].keys())

    if len(candidate_items) < 2:
        continue

    scores = {item: model.predict(uid, item) for item in candidate_items}
    predicted_order = sorted(scores, key=lambda x: scores[x], reverse=True)

    score = ndcg_at_k(test_lookup[uid], predicted_order, k=min(K, len(candidate_items)))
    ndcg_scores.append(score)

    # For coverage/LTE, still use full catalog recommendations
    recommendations[uid] = model.recommend(uid, top_k=K, exclude_seen=True)

mean_ndcg = float(np.mean(ndcg_scores))
print(f"Mean NDCG@{K} over {len(ndcg_scores)} users: {mean_ndcg:.4f}")

## 4. Catalog Coverage

**Catalog coverage** measures breadth: what fraction of all known items does the model
recommend to *at least one user*?

A model with low coverage is a **popularity bubble** — it routes most users through the same
small slice of popular items, leaving the rest of the catalog invisible. High coverage doesn't
guarantee quality, but near-zero coverage almost always signals popularity bias.

We use the open-world recommendations (top-K from the full catalog) — the right setup
for measuring how much of the catalog the model actually reaches.

In [ ]:
def catalog_coverage(recommendations: dict, all_items: set) -> float:
    recommended = set(item for recs in recommendations.values() for item, _ in recs)
    return len(recommended) / len(all_items)


all_items = set(model.item2idx.keys())
cov = catalog_coverage(recommendations, all_items)

unique_recommended = set(item for recs in recommendations.values() for item, _ in recs)
print(f"Catalog coverage : {cov:.4f} ({len(unique_recommended)} of {len(all_items)} items)")
print(f"\nNote: low coverage confirms popularity bias — the model recommends the same")
print(f"small set of popular items to nearly every user.")

## 5. Long-tail Exposure Rate

Phase 1 showed that **80% of items receive fewer than 20% of all ratings** — a classic long-tail
distribution. We use the same definition here: tail items = bottom 80% of items by rating count.

**Long-tail exposure rate** asks: of all recommendations produced, what fraction point to tail items?
A rate near 0 means the model is amplifying the popularity bias seen in the raw data.
A rate near 0.80 would mean the model is proportionally representing the tail.

This is the most direct connection between Phase 1 and Phase 2.

In [ ]:
# tail = bottom 80% of items by rating count (same definition as Phase 1)
item_counts = ratings_small.groupby('movieId')['rating'].count().reset_index()
item_counts.columns = ['movieId', 'count']
item_counts = item_counts.sort_values('count')

cutoff_idx = int(np.floor(0.80 * len(item_counts)))
tail_items  = set(item_counts.iloc[:cutoff_idx]['movieId'])
head_items  = set(item_counts.iloc[cutoff_idx:]['movieId'])

print(f"Total items : {len(item_counts)}")
print(f"Tail items  : {len(tail_items)} ({100*len(tail_items)/len(item_counts):.1f}%)")
print(f"Head items  : {len(head_items)} ({100*len(head_items)/len(item_counts):.1f}%)")

In [ ]:
def long_tail_exposure(recommendations: dict, tail_items: set) -> float:
    all_recs = [item for recs in recommendations.values() for item, _ in recs]
    if not all_recs:
        return 0.0
    return sum(1 for item in all_recs if item in tail_items) / len(all_recs)


lte = long_tail_exposure(recommendations, tail_items)

all_recs_flat = [item for recs in recommendations.values() for item, _ in recs]
n_tail_recs   = sum(1 for item in all_recs_flat if item in tail_items)
n_head_recs   = len(all_recs_flat) - n_tail_recs

print(f"Long-tail exposure rate : {lte:.4f}")
print(f"  tail recs : {n_tail_recs} / {len(all_recs_flat)}")
print(f"  head recs : {n_head_recs} / {len(all_recs_flat)}")

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(['Head (top 20%)', 'Tail (bottom 80%)'],
              [n_head_recs, n_tail_recs],
              color=['#e07b54', '#5b8db8'])
ax.bar_label(bars, fmt='%d')
ax.set_ylabel('# recommendations')
ax.set_title(f'Head vs Tail Recommendations (N={len(sampled_users)} users, K={K})')
plt.tight_layout()
plt.show()

## 6. Results Summary

In [ ]:
summary = pd.DataFrame([
    {'Metric': 'RMSE (test)',
     'Value': f'{test_rmse:.4f}',
     'Ideal': '→ 0',
     'Interpretation': 'Avg prediction error (rating scale 0.5–5.0)'},
    {'Metric': f'NDCG@{K} closed-world (mean, {len(ndcg_scores)} users)',
     'Value': f'{mean_ndcg:.4f}',
     'Ideal': '→ 1',
     'Interpretation': 'Ranking quality over each user\'s own test items'},
    {'Metric': 'Catalog coverage',
     'Value': f'{cov:.4f} ({len(unique_recommended)}/{len(all_items)} items)',
     'Ideal': '→ 1',
     'Interpretation': 'Fraction of catalog recommended to ≥1 user (open-world)'},
    {'Metric': 'Long-tail exposure rate',
     'Value': f'{lte:.4f}',
     'Ideal': '→ 0.80',
     'Interpretation': 'Fraction of recs pointing to tail items (80% of catalog)'},
])

summary.set_index('Metric', inplace=True)
summary

## 7. Key Observations

### Evaluation protocol matters

Open-world NDCG (ranking all ~16k items) was near zero — not because the model is bad, but
because with only a handful of test items per user the overlap probability is under 0.1% by chance.
Closed-world evaluation (ranking only each user's own test-rated items) gives a meaningful signal:
can the model tell the difference between a 5-star and a 2-star rating within the same user's history?

### Ranking vs prediction accuracy

The model achieves reasonable RMSE (~0.90) but the NDCG reveals whether it has learned a meaningful
preference order. Bias terms (μ + b_u + b_i) explain most of RMSE improvement — they capture
"this user rates high" / "this item is rated high" patterns without necessarily learning
fine-grained item preference order within a user.

### Does MF perpetuate popularity bias?

Yes — catalog coverage and long-tail exposure make it quantitative.
Phase 1 showed the top 20% of items get ~80% of ratings. If MF were neutral,
long-tail exposure should approach 0.80. The actual rate is much lower, and
catalog coverage (very few unique items recommended across all users) confirms
the model collapses to the same popular items for nearly everyone.

The mechanism is structural: popular items appear in more training pairs → more SGD updates
→ better-optimised latent factors and bias terms → they dominate at inference time.

### What would improve long-tail exposure?

1. **Popularity penalty** — subtract `α·log(count(i)+1)` from scores at inference (no retraining)
2. **Re-weighted loss** — up-weight tail interactions during SGD
3. **Item bias regularisation** — penalise large positive `b_i` terms
4. **Post-hoc re-ranking** — enforce a head/tail ratio per list (MMR)

### What's next

These findings motivate a controlled experiment: **does a popularity-penalised variant improve
long-tail exposure without unacceptable NDCG loss?**
The design for that experiment is in `docs/ab_test_design.md`.